# OpenClaw Observability with Opik

Adds full-stack observability to your OpenClaw agent: LLM traces, token costs, tool execution, and LLM-as-a-judge evaluation.

**Prerequisites:** OpenClaw gateway must be running. If it's not, run Part 5 of `openclaw_hackday.ipynb` first.

**Reference:** https://www.comet.com/site/blog/openclaw-observability/

In [12]:
import subprocess
import shutil
import time
import os
from dotenv import load_dotenv

load_dotenv(override=True)

# Auto-detect Docker sudo requirement
SUDO = ""
result = subprocess.run("docker info".split(), capture_output=True)
if result.returncode != 0:
    result = subprocess.run("sudo docker info".split(), capture_output=True)
    if result.returncode == 0:
        SUDO = "sudo "

OPENCLAW_COMPOSE = (
    f"{SUDO}docker compose -f openclaw-agent/docker-compose.yml"
    if subprocess.run(f"{SUDO}docker compose version".split(), capture_output=True).returncode == 0
    else f"{SUDO}docker-compose -f openclaw-agent/docker-compose.yml"
)

# Sync .env so the gateway picks up OPIK_API_KEY
shutil.copy(".env", "openclaw-agent/.env")
print(f"Keys synced. Using: {OPENCLAW_COMPOSE}")

# Start gateway if not running
health = subprocess.run(
    f"{OPENCLAW_COMPOSE} exec openclaw-gateway curl -sf http://localhost:18789/healthz".split(),
    capture_output=True
)
if health.returncode == 0:
    print("✅ Gateway already running.")
else:
    print("Starting OpenClaw gateway...")
    !{OPENCLAW_COMPOSE} up -d openclaw-gateway
    for i in range(15):
        health = subprocess.run(
            f"{OPENCLAW_COMPOSE} exec openclaw-gateway curl -sf http://localhost:18789/healthz".split(),
            capture_output=True
        )
        if health.returncode == 0:
            print("✅ Gateway is healthy!")
            break
        time.sleep(3)
        print(f"   Waiting... ({i+1})")

Keys synced. Using: docker compose -f openclaw-agent/docker-compose.yml
✅ Gateway already running.


---
## Step 1: Install the Opik Plugin

Downloads `@opik/opik-openclaw` from npm and installs it via the OpenClaw plugin system.

> **Note:** The ClawHub registry download for this package is currently broken. This cell uses a direct npm tarball workaround.

In [13]:
# Download npm tarball inside container, then install locally
!{OPENCLAW_COMPOSE} exec openclaw-gateway bash -c \
    "curl -sL https://registry.npmjs.org/@opik/opik-openclaw/-/opik-openclaw-0.2.10.tgz \
     -o /tmp/opik-openclaw.tgz && \
     node /app/openclaw.mjs plugins install /tmp/opik-openclaw.tgz"

---
## Step 2: Configure Opik

Set your Opik API key. Get one free at https://www.comet.com/opik

The key must be in your `.env` as `OPIK_API_KEY` — it's passed to the container via `docker-compose.yml`.

In [18]:
OPIK_API_KEY = os.getenv("OPIK_API_KEY", "")
OPIK_WORKSPACE = os.getenv("OPIK_WORKSPACE", "")
assert OPIK_API_KEY, "Set OPIK_API_KEY in .env"
assert OPIK_WORKSPACE, "Set OPIK_WORKSPACE in .env (your Comet username)"
print(f"Using Opik key: {OPIK_API_KEY[:8]}...")
print(f"Workspace: {OPIK_WORKSPACE}")

# Enable the plugin — the API key and workspace come from env vars passed via docker-compose.yml
!{OPENCLAW_COMPOSE} exec openclaw-gateway node /app/openclaw.mjs config set plugins.entries.opik-openclaw.config.enabled true --strict-json

print("\n✅ Plugin enabled. API key and workspace read from OPIK_API_KEY / OPIK_WORKSPACE env vars.")

---
## Step 3: Verify the Integration

In [19]:
# Note: 'API key: (not set)' in the status output is expected — the plugin reads
# OPIK_API_KEY from the environment directly, not from the config file.
# As long as OPIK_API_KEY is in your .env, traces will be exported correctly.
!{OPENCLAW_COMPOSE} exec openclaw-gateway node /app/openclaw.mjs opik status

---
## Step 4: Restart the Gateway

Required to activate the plugin.

In [20]:
!{OPENCLAW_COMPOSE} restart openclaw-gateway

print("Waiting for gateway...")
for i in range(15):
    health = subprocess.run(
        f"{OPENCLAW_COMPOSE} exec openclaw-gateway curl -sf http://localhost:18789/healthz".split(),
        capture_output=True
    )
    if health.returncode == 0:
        print("✅ Gateway healthy — Opik plugin active.")
        break
    time.sleep(3)
    print(f"   Waiting... ({i+1})")

---
## Step 5: Trigger a Trace

Send a message through a connected channel to generate traces in Opik.

> **Note:** The Opik plugin hooks into channel-based sessions (e.g. Telegram, Slack). If you have a channel connected, send a message there. The CLI `agent chat` command below may not produce traces, but you can use it to verify the agent is responding correctly.

In [21]:
!{OPENCLAW_COMPOSE} exec openclaw-gateway node /app/openclaw.mjs agent --agent main -m "What SEC filings have been submitted this week?"

print("\nIf you have a Telegram or Slack channel connected, send a message there to generate a trace.")
print("\n✅ Check your Opik dashboard:")
print("   https://app.comet.com/opik")


---
## What You Now Have

| Capability | Details |
| --- | --- |
| **Full trace capture** | Every LLM call, tool execution, memory recall, context assembly |
| **Token cost visibility** | Per-request, per-model cost breakdowns |
| **Conversation threading** | Multi-step reasoning traced end-to-end across sessions |
| **LLM-as-a-judge evaluation** | Hallucination detection, answer relevance, context precision |

Open your Opik dashboard to explore: https://app.comet.com/opik